# AI-RAG · Week 14 — Notebook 1 (Day 1)
## The Raw ReAct Agent + Failure-Oriented Design

The programme's **peak-density week** — the agent counterpart to building RAG from scratch. Day 1: build a **ReAct agent in ~80 lines of raw Python** (no framework), then name the failures it will face with a **5-category taxonomy**. Everything runs **offline** on a scripted mock LLM (no key, no cost).

> Code is lifted **verbatim** from the package (`src/agents/**`, `scripts/**`). Changes are notebook glue only: local data paths, dropped/redirected relative imports (code is inlined), removed CLI blocks — and the real-OpenAI model string is updated to `gpt-5-mini` to match the corrected deck (that path isn't exercised offline).

### Setup

In [1]:
# --- notebook setup (delivery glue only) -------------------------------------
# The whole package runs on a scripted mock LLM — no OpenAI/Tavily key, no cost.
# We ensure pydantic and write the offline web fixtures; the SQLite side-DB is built next.
try:
    import pydantic  # noqa: F401
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pydantic"])

from pathlib import Path
Path("data").mkdir(exist_ok=True)
Path("data/web_fixtures.json").write_text(r"""{
  "ai agents": [
    {"title": "State of AI Agents 2026", "snippet": "Agentic systems moved from demos to production in 2026, with tool-use reliability the key differentiator."},
    {"title": "Agent frameworks roundup", "snippet": "Raw loops and graph-based orchestration remain the two dominant patterns."}
  ],
  "openai api": [
    {"title": "OpenAI API pricing", "snippet": "gpt-4o-mini is positioned as the low-cost tier for high-volume workloads."}
  ],
  "cricket": [
    {"title": "Cricket World Cup — result", "snippet": "The most recent tournament concluded with a closely contested final."}
  ],
  "python libraries": [
    {"title": "Trending Python libraries", "snippet": "Data, LLM-tooling, and async libraries dominate this year's trending lists."}
  ],
  "announce": [
    {"title": "OpenAI announcements", "snippet": "Recent announcements focus on tool use, structured outputs, and cost reductions."}
  ],
  "_default": [
    {"title": "Web result", "snippet": "A relevant public-web snippet would appear here (offline fixture)."}
  ]
}""" + "\n")
print("wrote offline web fixtures · the agent runs on MockLLM (no key needed)")


wrote offline web fixtures · the agent runs on MockLLM (no key needed)


### Setup · the side-database (`scripts/seed_db.py`)

The SQL tool queries a small structured side-table; this builds it offline.

In [2]:
#!/usr/bin/env python3
"""scripts/seed_db.py — (re)create the capstone side database.

W13 gives the assistant structured data to query as a tool. This builds a small
SQLite database with two tables:

  document_metadata  — one row per document in the knowledge base (the structured
                       side-table the SQL tool queries)
  audit_log          — every SQL query the agent runs is logged here (the audit
                       rail for SQL-as-a-tool)

SQLite is local and offline, so the whole SQL-tool exercise runs with no network
and no credentials. Run this once (smoke_test.sh does it for you):

    python3 scripts/seed_db.py
"""
from __future__ import annotations

import sqlite3
from pathlib import Path

DB_PATH = Path("data/capstone_side.db")   # notebook: local file

# A small, realistic slice of the Enterprise Knowledge Assistant's corpus,
# across the three departments the capstone covers.
DOCS = [
    # source, title, doc_type, department, version, updated_at, chunk_count
    ("hr-001", "Leave & Time-Off Policy",          "policy",   "HR",      "v3", "2026-01-14", 42),
    ("hr-002", "Code of Conduct",                  "policy",   "HR",      "v2", "2025-11-02", 55),
    ("hr-003", "Remote Work Guidelines",           "guideline","HR",      "v4", "2026-02-20", 28),
    ("hr-004", "Onboarding Handbook",              "handbook", "HR",      "v1", "2025-06-30", 90),
    ("it-001", "Acceptable Use Policy",            "policy",   "IT",      "v5", "2026-01-30", 37),
    ("it-002", "Security Incident Runbook",        "runbook",  "IT",      "v2", "2026-02-11", 33),
    ("it-003", "VPN & Access Setup",               "guideline","IT",      "v3", "2025-12-05", 19),
    ("it-004", "Data Classification Standard",     "standard", "IT",      "v2", "2026-01-08", 24),
    ("fin-001","Expense Reimbursement Policy",     "policy",   "Finance", "v3", "2026-01-22", 31),
    ("fin-002","Procurement Guidelines",           "guideline","Finance", "v1", "2025-09-15", 40),
    ("fin-003","Travel Policy",                    "policy",   "Finance", "v2", "2025-10-19", 26),
    ("fin-004","Corporate Card Terms",             "terms",    "Finance", "v4", "2026-02-01", 18),
]


def seed(db_path: Path = DB_PATH) -> Path:
    db_path.parent.mkdir(parents=True, exist_ok=True)
    if db_path.exists():
        db_path.unlink()
    con = sqlite3.connect(db_path)
    cur = con.cursor()
    cur.execute("""
        CREATE TABLE document_metadata (
            id           INTEGER PRIMARY KEY,
            source       TEXT NOT NULL,
            title        TEXT NOT NULL,
            doc_type     TEXT NOT NULL,
            department   TEXT NOT NULL,
            version      TEXT NOT NULL,
            updated_at   TEXT NOT NULL,
            chunk_count  INTEGER NOT NULL
        )""")
    cur.executemany(
        "INSERT INTO document_metadata "
        "(source,title,doc_type,department,version,updated_at,chunk_count) "
        "VALUES (?,?,?,?,?,?,?)", DOCS)
    cur.execute("""
        CREATE TABLE audit_log (
            id         INTEGER PRIMARY KEY AUTOINCREMENT,
            ts         TEXT NOT NULL,
            actor      TEXT NOT NULL,
            request    TEXT,
            sql        TEXT,
            decision   TEXT NOT NULL,
            row_count  INTEGER,
            error      TEXT
        )""")
    con.commit()
    con.close()
    return db_path

seed()
print('side-DB built:', DB_PATH)


side-DB built: data/capstone_side.db


## 1 · The LLM behind the agent — `src/agents/mock_llm.py`

The agent talks to a tiny interface: `llm.complete(messages, tools) -> LLMResponse` (a tool call **or** a final answer). Two implementations: **`MockLLM`** — deterministic and offline, driven by a *script* (for tests) or keyword-routing (for demos) — and **`OpenAILLM`** (the real thing). The whole package runs on `MockLLM`, which is *why* it works with no credentials. (Mocking the LLM is the week's testing lesson — Notebook 2.)

In [3]:
"""src/agents/mock_llm.py — the LLM behind the agent, and how we mock it.

The single most useful testing skill this week is **mocking the LLM**. A real LLM
call is slow, costs money, and is non-deterministic — useless in a test. So the
agent talks to a tiny interface:

    llm.complete(messages, tools) -> LLMResponse   # either a tool call or a final answer

Two implementations:
  MockLLM  — deterministic and offline. Give it a *script* (a fixed list of
             responses, for tests) or let it keyword-route one tool call then a
             final answer (for offline runs). No network, no key, milliseconds.
  OpenAILLM — the real thing (OpenAI function calling). Needs OPENAI_API_KEY.

Everything in this package runs on MockLLM, which is why the whole thing — agent,
defenses, the five failure scenarios, the test suite — works with no credentials.
"""
from __future__ import annotations

import json
import re
from dataclasses import dataclass
from typing import Callable

# Tool is defined in the cell above


@dataclass
class LLMResponse:
    kind: str                      # "tool_use" | "final"
    tool_name: str | None = None
    tool_args: dict | None = None
    content: str | None = None

    @staticmethod
    def tool(name: str, **args) -> "LLMResponse":
        return LLMResponse("tool_use", tool_name=name, tool_args=args)

    @staticmethod
    def final(content: str) -> "LLMResponse":
        return LLMResponse("final", content=content)


class MockLLM:
    """A deterministic stand-in for a function-calling LLM.

    - Scripted:  MockLLM(script=[LLMResponse.tool('calculate', expression='2+2'),
                                 LLMResponse.final('It is 4.')])
                 returns those in order (extra calls repeat the last one).
    - Keyword:   MockLLM()  → routes ONE tool call by keyword, then a final answer
                 synthesised from the observation. Good for offline demos.
    """

    def __init__(self, script: list[LLMResponse] | None = None,
                 router: Callable[[list[dict], list[Tool]], LLMResponse] | None = None):
        self.script = list(script) if script else None
        self.router = router
        self.calls = 0

    def complete(self, messages: list[dict], tools: list[Tool]) -> LLMResponse:
        self.calls += 1
        if self.script is not None:
            return self.script.pop(0) if self.script else LLMResponse.final(
                "(no more scripted responses)")
        if self.router is not None:
            return self.router(messages, tools)
        return self._keyword_route(messages, tools)

    # ── default keyword routing: one tool call, then a final answer ──────────
    def _keyword_route(self, messages: list[dict], tools: list[Tool]) -> LLMResponse:
        # If a tool has already produced an observation, answer from it.
        obs = _last_observation(messages)
        if obs is not None:
            return LLMResponse.final(f"Based on the tool result: {obs}")
        q = _last_user(messages).lower()
        names = {t.name for t in tools}
        if "calculate" in names and re.search(r"\d\s*[-+*/%]\s*\d|percent|%|how much is", q):
            return LLMResponse.tool("calculate", expression=_extract_expr(q))
        sql_kw = ("how many doc", "which department", "what version", "list the", "count",
                  "how many hr", "how many it", "how many finance", "updated", "chunk")
        if "query_sql" in names and any(k in q for k in sql_kw):
            return LLMResponse.tool("query_sql",
                                    sql="SELECT COUNT(*) AS total_documents FROM document_metadata",
                                    purpose="metadata question")
        web_kw = ("latest", "current", "today", "news", "price", "recent", "2026", "who won")
        if "search_web" in names and any(k in q for k in web_kw):
            return LLMResponse.tool("search_web", query=_last_user(messages), max_results=3)
        return LLMResponse.final("I can answer that directly: (no tool needed).")


# ─── real OpenAI adapter (needs a key + network) ────────────────────────────
class OpenAILLM:
    def __init__(self, model: str = "gpt-5-mini"):
        from openai import OpenAI
        self.client = OpenAI()
        self.model = model

    def complete(self, messages: list[dict], tools: list[Tool]) -> LLMResponse:
        resp = self.client.chat.completions.create(
            model=self.model, messages=messages,
            tools=[t.openai_schema() for t in tools], tool_choice="auto")
        msg = resp.choices[0].message
        if msg.tool_calls:
            call = msg.tool_calls[0]
            return LLMResponse.tool(call.function.name, **json.loads(call.function.arguments or "{}"))
        return LLMResponse.final(msg.content or "")


# ─── tiny helpers ───────────────────────────────────────────────────────────
def _last_user(messages: list[dict]) -> str:
    for m in reversed(messages):
        if m.get("role") == "user":
            return str(m.get("content", ""))
    return ""


def _last_observation(messages: list[dict]) -> str | None:
    for m in reversed(messages):
        if m.get("role") == "tool":
            return str(m.get("content", ""))
    return None


def _extract_expr(q: str) -> str:
    m = re.search(r"[0-9][0-9\.\s\+\-\*/%\(\)]*[0-9\)]", q)
    return m.group(0).strip() if m else "0"


**A scripted mock returns its responses in order:**

In [4]:
llm = MockLLM(script=[LLMResponse.tool('calculate', expression='2+2'),
                      LLMResponse.final('It is 4.')])
m = [{'role': 'user', 'content': 'what is 2+2'}]
r1 = llm.complete(m, []);  print('call 1 →', r1.kind, r1.tool_name, r1.tool_args)
r2 = llm.complete(m, []);  print('call 2 →', r2.kind, r2.content)

call 1 → tool_use calculate {'expression': '2+2'}
call 2 → final It is 4.


## 2 · The tools

Carried from W13 (`base`, `calculate`, `query_sql`, `search_web`) plus one new **write** tool, `send_email` — inlined so the agent can run. `send_email` has a side effect, which is why the permissioning defense exists (Notebook 2).

In [5]:
"""src/agents/tools/base.py — the Tool contract.

A tool an LLM can call reliably is four things done well (the W13 principles):
  1. a good NAME       — a verb, specific, snake_case
  2. a typed SCHEMA    — a Pydantic model: required fields, sensible defaults
  3. a DESCRIPTION     — written for the *model*, saying WHEN to use the tool
  4. IDEMPOTENCY        — declared, so the loop knows if a retry is safe

Every tool subclasses `Tool`, sets those four, and implements `run`. The same
definition exports to both OpenAI function-calling and Anthropic tool-use schemas.
"""
from __future__ import annotations

from typing import Any

from pydantic import BaseModel


class Tool:
    name: str = ""
    description: str = ""
    Args: type[BaseModel] = BaseModel
    idempotent: bool = True          # is calling this twice with the same args safe?
    has_side_effects: bool = False   # does it change external state?

    def run(self, **kwargs: Any) -> str:
        raise NotImplementedError

    # ── validation + execution ────────────────────────────────────────────
    def call(self, arguments: dict[str, Any]) -> str:
        """Validate arguments against the schema, then run."""
        args = self.Args(**(arguments or {}))
        return self.run(**args.model_dump())

    # ── provider schema exports ───────────────────────────────────────────
    def parameters_schema(self) -> dict:
        return self.Args.model_json_schema()

    def openai_schema(self) -> dict:
        return {"type": "function",
                "function": {"name": self.name,
                             "description": self.description,
                             "parameters": self.parameters_schema()}}

    def anthropic_schema(self) -> dict:
        return {"name": self.name,
                "description": self.description,
                "input_schema": self.parameters_schema()}


In [6]:
"""src/agents/tools/calculate.py — a computation tool.

Deterministic, offline, and safe: it evaluates an arithmetic expression with a
small AST walker (no `eval`), so an LLM can hand it a calculation and get a
reliable number back. Idempotent and side-effect-free — the easy end of the
tool spectrum, and a good first example of a well-formed tool.
"""
from __future__ import annotations

import ast
import operator

from pydantic import BaseModel, Field

# Tool is defined in the cell above

_OPS = {
    ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul,
    ast.Div: operator.truediv, ast.Mod: operator.mod, ast.Pow: operator.pow,
    ast.USub: operator.neg, ast.UAdd: operator.pos, ast.FloorDiv: operator.floordiv,
}


def _eval(node: ast.AST) -> float:
    if isinstance(node, ast.Expression):
        return _eval(node.body)
    if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
        return node.value
    if isinstance(node, ast.BinOp) and type(node.op) in _OPS:
        return _OPS[type(node.op)](_eval(node.left), _eval(node.right))
    if isinstance(node, ast.UnaryOp) and type(node.op) in _OPS:
        return _OPS[type(node.op)](_eval(node.operand))
    raise ValueError("unsupported expression — only + - * / % ** ( ) and numbers")


class CalculateArgs(BaseModel):
    expression: str = Field(
        ..., description="An arithmetic expression, e.g. '1350 * 0.18' or '(42 + 8) / 2'.")


class CalculateTool(Tool):
    name = "calculate"
    description = ("Evaluate an arithmetic expression and return the numeric result. "
                   "Use this whenever a question needs exact math — sums, differences, "
                   "products, division, percentages, or powers. Do NOT use it to look up "
                   "facts; it only does arithmetic on numbers you provide.")
    Args = CalculateArgs
    idempotent = True
    has_side_effects = False

    def run(self, expression: str) -> str:
        try:
            result = _eval(ast.parse(expression, mode="eval"))
        except Exception as e:  # noqa: BLE001 - report cleanly to the agent
            return f"calculate error: {e}"
        if isinstance(result, float) and result.is_integer():
            result = int(result)
        return f"{expression} = {result}"


In [7]:
"""src/agents/tools/query_sql.py — SQL as a tool, safely.

Letting an LLM run SQL is powerful and dangerous. This tool ships the three rails
every SQL-as-a-tool needs in production:

  1. AUDIT LOGGING     — every query (allowed or denied) is written to audit_log
  2. READ-ONLY DEFAULT  — only a single SELECT / WITH…SELECT is permitted
  3. CONSTRAINED UPDATES — writes/DDL (INSERT/UPDATE/DELETE/DROP/…) are denied
                          outright (write access is a W18 topic, gated behind
                          injection defenses and a human-approval step)

Everything runs against a local SQLite database, so it works offline with no
credentials. The migration path to Postgres is a config change, not a rewrite
(see docs/runbooks/sql_as_a_tool_safety.md).
"""
from __future__ import annotations

import re
import sqlite3
from datetime import datetime, timezone
from pathlib import Path

from pydantic import BaseModel, Field

# Tool is defined in the cell above

DB_PATH = Path("data/capstone_side.db")   # notebook: local file (built in setup)

# Statements that change data or schema — denied by the read-only rail.
_FORBIDDEN = {"insert", "update", "delete", "drop", "alter", "create", "replace",
              "truncate", "attach", "detach", "pragma", "vacuum", "reindex",
              "grant", "revoke", "begin", "commit"}
_MAX_ROWS = 50


def _strip_comments(sql: str) -> str:
    sql = re.sub(r"--[^\n]*", " ", sql)
    sql = re.sub(r"/\*.*?\*/", " ", sql, flags=re.S)
    return sql.strip()


def is_read_only(sql: str) -> tuple[bool, str]:
    """Return (ok, reason). Only a single SELECT/WITH…SELECT is allowed."""
    clean = _strip_comments(sql)
    if not clean:
        return False, "empty query"
    statements = [s for s in clean.split(";") if s.strip()]
    if len(statements) > 1:
        return False, "multiple statements are not allowed"
    first = statements[0].lstrip().split(None, 1)[0].lower()
    if first not in ("select", "with"):
        return False, f"only SELECT queries are allowed (got '{first.upper()}')"
    words = set(re.findall(r"[a-z_]+", clean.lower()))
    hit = words & _FORBIDDEN
    if hit:
        return False, f"write/DDL keyword not permitted: {', '.join(sorted(hit))}"
    return True, "ok"


class QuerySqlArgs(BaseModel):
    sql: str = Field(..., description="A single read-only SQL SELECT query over the "
                                      "document_metadata table.")
    purpose: str = Field("", description="One phrase describing what the query is for "
                                         "(recorded in the audit log).")


class QuerySqlTool(Tool):
    name = "query_sql"
    description = (
        "Run a read-only SQL SELECT against the knowledge-base metadata database and "
        "return the rows. Use this for questions about the DOCUMENTS THEMSELVES — how "
        "many documents exist, which department owns them, their type, version, "
        "chunk_count, or when they were last updated. Table: document_metadata"
        "(id, source, title, doc_type, department, version, updated_at, chunk_count). "
        "Do NOT use it to answer questions about document CONTENTS — only their metadata.")
    Args = QuerySqlArgs
    idempotent = True            # SELECTs don't change state
    has_side_effects = False

    def __init__(self, db_path: Path | str = DB_PATH):
        self.db_path = Path(db_path)

    def _audit(self, request: str, sql: str, decision: str,
               row_count: int | None, error: str | None) -> None:
        con = sqlite3.connect(self.db_path)
        con.execute(
            "INSERT INTO audit_log (ts, actor, request, sql, decision, row_count, error) "
            "VALUES (?,?,?,?,?,?,?)",
            (datetime.now(timezone.utc).isoformat(timespec="seconds"), "agent",
             request, sql, decision, row_count, error))
        con.commit()
        con.close()

    def run(self, sql: str, purpose: str = "") -> str:
        ok, reason = is_read_only(sql)
        if not ok:
            self._audit(purpose, sql, "denied", None, reason)
            return (f"query_sql DENIED: {reason}. This tool is read-only; write access "
                    f"is not available. (Logged to the audit trail.)")
        try:
            con = sqlite3.connect(self.db_path)
            cur = con.execute(sql)
            rows = cur.fetchmany(_MAX_ROWS)
            cols = [d[0] for d in cur.description] if cur.description else []
            con.close()
        except Exception as e:  # noqa: BLE001
            self._audit(purpose, sql, "error", None, str(e))
            return f"query_sql error: {e}"
        self._audit(purpose, sql, "allowed", len(rows), None)
        if not rows:
            return "query_sql: (0 rows)"
        header = " | ".join(cols)
        body = "\n".join(" | ".join(str(v) for v in r) for r in rows)
        more = "" if len(rows) < _MAX_ROWS else f"\n… (capped at {_MAX_ROWS} rows)"
        return f"{header}\n{'-' * len(header)}\n{body}{more}"


In [8]:
"""src/agents/tools/search_web.py — an external-API tool (web search).

Wraps a web-search API (Tavily) so the agent can reach information beyond the
knowledge base — current events, prices, anything not in the corpus. This is the
"API-calling agent" pattern: the LLM decides it needs fresh external info and
calls out.

Web search has real cost and latency, so the description tells the model to use
it only when the answer isn't in the corpus. For offline runs and tests, the tool
falls back to a local fixture file, so the whole thing works with no network and
no TAVILY_API_KEY.
"""
from __future__ import annotations

import json
import os
from pathlib import Path

from pydantic import BaseModel, Field

# Tool is defined in the cell above

FIXTURES_PATH = Path("data/web_fixtures.json")   # notebook: local file (written in setup)


class SearchWebArgs(BaseModel):
    query: str = Field(..., description="A concise web-search query.")
    max_results: int = Field(3, ge=1, le=5, description="How many results to return.")


class SearchWebTool(Tool):
    name = "search_web"
    description = (
        "Search the public web and return short result snippets. Use this ONLY when the "
        "answer is not in the internal knowledge base — for current events, live figures, "
        "recent releases, or anything outside company documents. It has cost and latency, "
        "so prefer the internal tools when they can answer. Returns titles and snippets, "
        "not full pages.")
    Args = SearchWebArgs
    idempotent = True            # reading search results doesn't change state
    has_side_effects = False

    def __init__(self, offline: bool | None = None):
        # offline when explicitly set, or when there is no API key available
        self.offline = (not os.environ.get("TAVILY_API_KEY")) if offline is None else offline

    def _fixture(self, query: str, k: int) -> str:
        try:
            data = json.loads(FIXTURES_PATH.read_text())
        except Exception:
            data = {"_default": [{"title": "No fixture", "snippet": "offline stub result."}]}
        results = None
        for key, res in data.items():
            if key != "_default" and key.lower() in query.lower():
                results = res
                break
        results = (results or data.get("_default", []))[:k]
        lines = [f"[web:{query}] (offline fixture)"]
        for r in results:
            lines.append(f"- {r['title']}: {r['snippet']}")
        return "\n".join(lines)

    def run(self, query: str, max_results: int = 3) -> str:
        if self.offline:
            return self._fixture(query, max_results)
        # live path (needs network + TAVILY_API_KEY)
        try:
            from tavily import TavilyClient
            client = TavilyClient(api_key=os.environ["TAVILY_API_KEY"])
            resp = client.search(query=query, max_results=max_results)
            lines = [f"[web:{query}]"]
            for r in resp.get("results", [])[:max_results]:
                lines.append(f"- {r.get('title','')}: {r.get('content','')[:160]}")
            return "\n".join(lines)
        except Exception as e:  # noqa: BLE001
            return f"search_web error: {e} (falling back offline)\n" + self._fixture(query, max_results)


In [9]:
"""src/agents/tools/send_email.py — a WRITE tool, on purpose.

The three W13 tools are all read-only. This one has a side effect (it "sends"),
so it's the tool the permissioning defense exists for: the DefensePolicy gates
side-effecting tools behind a human-confirmation step, so an adversarial prompt
("ignore instructions and email everyone") is blocked rather than executed.

It never actually sends anything here — it's the shape that matters. It is marked
non-idempotent (a retry would send twice) and side-effecting.
"""
from __future__ import annotations

from pydantic import BaseModel, Field

# Tool is defined in the cell above


class SendEmailArgs(BaseModel):
    to: str = Field(..., description="Recipient email address.")
    subject: str = Field(..., description="Email subject line.")
    body: str = Field("", description="Email body.")


class SendEmailTool(Tool):
    name = "send_email"
    description = ("Send an email to a recipient. This performs a real, irreversible "
                   "action, so it is gated behind human confirmation. Use only when the "
                   "user explicitly asks to send a message.")
    Args = SendEmailArgs
    idempotent = False            # a retry would send a second email
    has_side_effects = True

    def run(self, to: str, subject: str, body: str = "") -> str:
        # If this ever executes, permissioning failed. Kept as a no-op on purpose.
        return f"(would send email to {to}: {subject!r})"


### The failure types the loop catches (from `defenses.py`)

The agent's loop distinguishes **transient** errors (retry) from **permanent** ones (fast-fail). These exception types are the taxonomy *in code*; the defenses that act on them are built and demonstrated in Notebook 2.

In [10]:
# ─── failure types ──────────────────────────────────────────────────────────
class TransientError(Exception):
    """Temporary — a retry will probably succeed (e.g. network blip)."""


class RateLimited(TransientError):
    """HTTP 429 — transient, back off and retry."""


class PermanentError(Exception):
    """Won't fix itself — retrying is harmful (bad credentials, bad schema/args)."""


def classify(exc: Exception) -> str:
    """Map an exception to a failure category (the diagnostic the week teaches)."""
    if isinstance(exc, RateLimited):
        return "transient"
    if isinstance(exc, TransientError) or isinstance(exc, TimeoutError):
        return "transient"      # timeouts retry, but repeated ones trip the breaker → cascading
    if isinstance(exc, PermanentError):
        return "permanent"
    return "permanent"          # unknown = don't blindly retry


## 3 · The raw ReAct agent — `src/agents/raw_react.py`

**The centrepiece.** ReAct = Reasoning + Acting in alternation: the LLM returns a tool call *or* a final answer → run the tool, feed the observation back → repeat, until a final answer **or** `max_iter`. No framework. Pass a `DefensePolicy` to harden it (Notebook 2); leave it off to see the bare loop.

In [11]:
"""src/agents/raw_react.py — a ReAct agent in ~80 lines of raw Python.

ReAct = Reasoning + Acting in alternation. The loop:

    1. the LLM looks at the question + tools and returns a tool call OR a final answer
    2. if a tool call → run the tool, feed the observation back
    3. repeat, until a final answer OR max_iterations is hit

No framework. The max-iteration guard is non-negotiable — without it, one bad
model response can loop forever. Pass a DefensePolicy to harden every tool call
(retries, circuit breaker, permissioning, graceful degradation); leave it off to
see the bare loop.
"""
from __future__ import annotations

import json
from dataclasses import dataclass, field
from typing import Any

# DefensePolicy, MockLLM, Tool are defined in the cells above


@dataclass
class Step:
    tool: str
    args: dict
    observation: str
    status: str = "ok"           # ok | degraded | blocked


@dataclass
class AgentResult:
    question: str
    answer: str
    steps: list[Step] = field(default_factory=list)
    iterations: int = 0
    stop_reason: str = "final"   # final | max_iter
    def to_dict(self) -> dict:
        return {"question": self.question, "answer": self.answer,
                "iterations": self.iterations, "stop_reason": self.stop_reason,
                "steps": [s.__dict__ for s in self.steps]}


SYSTEM = ("You are a helpful assistant that can call tools. Think about whether a "
          "tool is needed; call one if so, otherwise answer directly. Never invent "
          "tool results.")


class ReActAgent:
    def __init__(self, llm: Any, tools: list[Tool], max_iter: int = 5,
                 defenses: DefensePolicy | None = None):
        self.llm = llm
        self.tools = tools
        self.by_name = {t.name: t for t in tools}
        self.max_iter = max_iter
        self.defenses = defenses

    def _execute(self, name: str, args: dict) -> tuple[str, str]:
        tool = self.by_name.get(name)
        if tool is None:
            return f"[unknown tool '{name}']", "degraded"
        # PermanentError, RateLimited, TransientError are defined above

        def call() -> str:
            try:
                return tool.call(args or {})
            except (RateLimited, TransientError, TimeoutError):
                raise                             # let the policy retry / trip the breaker
            except PermanentError:
                raise                             # already classified
            except Exception as e:                # bad/hallucinated args, bad schema = permanent
                raise PermanentError(str(e))

        if self.defenses:
            return self.defenses.guard(name, call)
        return call(), "ok"                       # bare loop (no defenses)

    def run(self, question: str) -> AgentResult:
        result = AgentResult(question=question, answer="")
        messages = [{"role": "system", "content": SYSTEM},
                    {"role": "user", "content": question}]
        for i in range(self.max_iter):
            result.iterations = i + 1
            resp = self.llm.complete(messages, self.tools)
            if resp.kind == "final":
                result.answer = resp.content or ""
                result.stop_reason = "final"
                return result
            # tool call
            name, args = resp.tool_name, resp.tool_args or {}
            obs, status = self._execute(name, args)
            result.steps.append(Step(tool=name, args=args, observation=obs, status=status))
            call_id = f"call_{i}"
            messages.append({"role": "assistant", "content": None,
                             "tool_calls": [{"id": call_id, "type": "function",
                                             "function": {"name": name,
                                                          "arguments": json.dumps(args)}}]})
            messages.append({"role": "tool", "tool_call_id": call_id, "content": obs})
        # max-iteration guard fired
        result.stop_reason = "max_iter"
        result.answer = ("(stopped after the maximum number of steps — I couldn't reach a "
                         "final answer safely.)")
        return result


# ─── assembly ───────────────────────────────────────────────────────────────
def default_tools(offline: bool = True) -> list[Tool]:
    # the four Tool classes are defined in the cells above
    return [SearchWebTool(offline=offline), QuerySqlTool(), CalculateTool(), SendEmailTool()]


def build_agent(offline: bool = True, use_llm: bool = False, hardened: bool = True,
                max_iter: int = 5) -> ReActAgent:
    """Default: offline MockLLM + hardened with the DefensePolicy."""
    tools = default_tools(offline=offline)
    if use_llm:
        # OpenAILLM is defined above
        llm: Any = OpenAILLM()
    else:
        llm = MockLLM()
    policy = DefensePolicy() if hardened else None
    return ReActAgent(llm, tools, max_iter=max_iter, defenses=policy)


**Run it in bare mode** (no defenses) and read the reason→act trace:

In [12]:
agent = build_agent(offline=True, hardened=False)   # the raw loop, no defenses
r = agent.run('How many documents are there?')
print('Q:', r.question)
for s in r.steps:
    print(f'  {s.tool}({s.args}) → [{s.status}] {s.observation[:55]}')
print('A:', r.answer)
print('stop_reason:', r.stop_reason, '| iterations:', r.iterations)

Q: How many documents are there?
  query_sql({'sql': 'SELECT COUNT(*) AS total_documents FROM document_metadata', 'purpose': 'metadata question'}) → [ok] total_documents
---------------
12
A: Based on the tool result: total_documents
---------------
12
stop_reason: final | iterations: 2


## 4 · The `max_iter` guard — non-negotiable

Without a step budget, one bad model response loops forever. Give the agent a model that **never** returns a final answer, and watch the guard stop it safely (this is the gauntlet's *control* scenario, S6):

In [13]:
never_finishes = MockLLM(router=lambda m, t: LLMResponse.tool('calculate', expression='1+1'))
r = ReActAgent(never_finishes, default_tools(offline=True), max_iter=5).run('loop')
print('stop_reason:', r.stop_reason, '| iterations:', r.iterations)
print('answer:', r.answer)

stop_reason: max_iter | iterations: 5
answer: (stopped after the maximum number of steps — I couldn't reach a final answer safely.)


## 5 · Failure-Oriented Design — the 5-category taxonomy

**Assume failure; design for it.** Before hardening, name what can go wrong. Five failure categories, each with its defense:

| Category | Example | Defense |
|---|---|---|
| **Transient** | a 429 / network blip | retry with backoff |
| **Permanent** | bad args / bad schema | fast-fail (do *not* retry) |
| **Silent** | a wrong-but-valid result | validation / sanity checks |
| **Cascading** | a timeout floods downstream | circuit breaker |
| **Adversarial** | injection / misuse | tool permissioning (+ sanitisation) |

These five categories return in W18 (injection defense), W22, and W25. Notebook 2 implements each defense, then proves the agent survives all five in the gauntlet — **six scenarios in total: the five injected failures, plus the runaway-loop guard.**

## Recap

A ReAct agent is one small loop — reason, act, observe, repeat — with a non-negotiable `max_iter` guard, and it runs offline on a mock LLM. You can now name any failure with the 5-category taxonomy. Next, **Notebook 2** implements the five defenses, runs the failure gauntlet, and mocks the LLM to test the agent properly.